In [4]:
# ============================================================
# OGPO EDA PIPELINE
# ============================================================
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')


In [5]:
# ============================================================
# 1. ЗАГРУЗКА ДАННЫХ
# ============================================================
print("=" * 50)
print("1. ЗАГРУЗКА")
print("=" * 50)

df = pl.scan_csv(
    "final_dataset/train.csv",
    infer_schema_length=10000,
    schema_overrides={"bonus_malus": pl.Utf8},  # читаем как строку
    null_values=["", "NA", "N/A", "null", "None"]
).collect()



1. ЗАГРУЗКА


In [6]:
print(df["bonus_malus"].unique().sort())

shape: (16,)
Series: 'bonus_malus' [str]
[
	null
	"0"
	"1"
	"10"
	"11"
	…
	"6"
	"7"
	"8"
	"9"
	"M"
]


In [7]:


# Колонки по типам
TARGET = "is_claim"
LEAKAGE_COLS = ["claim_amount", "claim_cnt"]  # НЕЛЬЗЯ в фичи!
ID_COLS = ["unique_id", "contract_number", "insurer_iin", "driver_iin", "car_number"]
DATE_COLS = ["operation_date"]
SCORE_COLS = [c for c in df.columns if c.startswith("SCORE_")]
CAT_COLS = ["region_name", "vehicle_type_name", "mark", "model",
            "age_experience_name", "is_individual_person_name", "is_residence_name"]
NUM_COLS = ["premium", "premium_wo_term", "bonus_malus", "experience_year",
            "car_age", "car_year", "engine_volume", "engine_power"]

print(f"\nSCORE колонок: {len(SCORE_COLS)}")
print(f"Числовых фич: {len(NUM_COLS)}")
print(f"Категориальных фич: {len(CAT_COLS)}")



SCORE колонок: 128
Числовых фич: 8
Категориальных фич: 7


In [8]:

# ============================================================
# 2. БАЗОВАЯ ИНФОРМАЦИЯ
# ============================================================
print("\n" + "=" * 50)
print("2. БАЗОВАЯ ИНФОРМАЦИЯ")
print("=" * 50)

df_pd = df.to_pandas()

# Пропуски
missing = df_pd.isnull().sum()
missing_pct = (missing / len(df_pd) * 100).round(2)
missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).query("missing_count > 0").sort_values("missing_pct", ascending=False)

print("\nПРОПУСКИ:")
print(missing_df.to_string())

# Дубликаты
print(f"\nДубликаты строк: {df_pd.duplicated().sum()}")
print(f"Уникальных полисов: {df_pd['contract_number'].nunique()}")
print(f"Всего строк: {len(df_pd)}")
print(f"Среднее водителей на полис: {len(df_pd) / df_pd['contract_number'].nunique():.2f}")



2. БАЗОВАЯ ИНФОРМАЦИЯ



ПРОПУСКИ:
                 missing_count  missing_pct
SCORE_12_4              569246        99.95
SCORE_12_6              569155        99.94
SCORE_12_5              568355        99.80
SCORE_12_1              567738        99.69
SCORE_12_2              567552        99.66
SCORE_12_3              565842        99.36
claim_amount            558414        98.05
claim_cnt               558414        98.05
SCORE_12_9_1            547110        96.07
SCORE_11_9              486576        85.44
SCORE_11_1              486576        85.44
SCORE_11_11             486576        85.44
SCORE_11_10             486576        85.44
SCORE_11_12             486576        85.44
SCORE_11_5              486576        85.44
SCORE_11_4              486576        85.44
SCORE_11_3              486576        85.44
SCORE_11_2              486576        85.44
SCORE_11_8              486576        85.44
SCORE_11_6              486576        85.44
SCORE_11_7              486576        85.44
SCORE_7_3            

In [9]:

# ============================================================
# 3. АНАЛИЗ ТАРГЕТА (is_claim)
# ============================================================
print("\n" + "=" * 50)
print("3. АНАЛИЗ ТАРГЕТА is_claim")
print("=" * 50)

target_counts = df_pd[TARGET].value_counts()
target_pct = df_pd[TARGET].value_counts(normalize=True) * 100
print(f"\nРаспределение таргета:")
print(f"  0 (нет выплат): {target_counts[0]:,}  ({target_pct[0]:.2f}%)")
print(f"  1 (есть выплата): {target_counts[1]:,}  ({target_pct[1]:.2f}%)")
print(f"  Дисбаланс: 1:{target_counts[0]//target_counts[1]}")

# Выплаты по полисам (агрегировано)
policy_claims = df_pd.groupby("contract_number")[TARGET].max()
print(f"\nПолисов с выплатами: {policy_claims.sum():,} из {len(policy_claims):,} ({policy_claims.mean()*100:.2f}%)")



3. АНАЛИЗ ТАРГЕТА is_claim

Распределение таргета:
  0 (нет выплат): 558,414  (98.05%)
  1 (есть выплата): 11,094  (1.95%)
  Дисбаланс: 1:50

Полисов с выплатами: 3,575 из 180,635 (1.98%)


In [10]:

# ============================================================
# 4. АНАЛИЗ ФИНАНСОВЫХ ПЕРЕМЕННЫХ
# ============================================================
print("\n" + "=" * 50)
print("4. ФИНАНСОВЫЕ ПЕРЕМЕННЫЕ")
print("=" * 50)

# Уникальные значения на уровне полиса
policy_df = df_pd.drop_duplicates(subset="contract_number")
print(f"\nАнализ на уровне полисов ({len(policy_df):,} полисов):")
for col in ["premium", "premium_wo_term", "claim_amount"]:
    if col in policy_df.columns:
        print(f"\n{col}:")
        print(f"  mean={policy_df[col].mean():,.0f}  median={policy_df[col].median():,.0f}"
              f"  min={policy_df[col].min():,.0f}  max={policy_df[col].max():,.0f}")
        print(f"  zeros: {(policy_df[col]==0).sum():,} ({(policy_df[col]==0).mean()*100:.1f}%)")

# Loss Ratio текущий
total_claims = policy_df["claim_amount"].sum()
total_premium = policy_df["premium_wo_term"].sum()
current_lr = total_claims / total_premium
print(f"\nТекущий Loss Ratio: {current_lr:.2%}  (цель: ~70%)")



4. ФИНАНСОВЫЕ ПЕРЕМЕННЫЕ

Анализ на уровне полисов (180,635 полисов):

premium:
  mean=13,186  median=11,431  min=147  max=116,896
  zeros: 0 (0.0%)

premium_wo_term:
  mean=10,645  median=9,526  min=0  max=116,896
  zeros: 2,294 (1.3%)

claim_amount:
  mean=663,414  median=376,409  min=8,500  max=23,284,342
  zeros: 0 (0.0%)

Текущий Loss Ratio: 123.34%  (цель: ~70%)


In [11]:

# ============================================================
# 5. АНАЛИЗ ЧИСЛОВЫХ ПРИЗНАКОВ vs ТАРГЕТ
# ============================================================
print("\n" + "=" * 50)
print("5. ЧИСЛОВЫЕ ПРИЗНАКИ vs ТАРГЕТ")
print("=" * 50)
for col in NUM_COLS:
    if col not in df_pd.columns:
        continue
    
    # Принудительно конвертируем в числа, строки → NaN
    col_data = pd.to_numeric(df_pd[col], errors='coerce')
    
    group0 = col_data[df_pd[TARGET] == 0].dropna()
    group1 = col_data[df_pd[TARGET] == 1].dropna()
    
    if len(group0) == 0 or len(group1) == 0:
        print(f"{col:20s} | ПРОПУЩЕНО (пустая группа)")
        continue
    
    stat, pval = stats.mannwhitneyu(group0, group1, alternative='two-sided')
    significance = "***" if pval < 0.001 else ("**" if pval < 0.01 else ("*" if pval < 0.05 else ""))
    
    print(f"{col:20s} | mean_0={group0.mean():8.2f} | mean_1={group1.mean():8.2f} "
          f"| p={pval:.4f} {significance}")



5. ЧИСЛОВЫЕ ПРИЗНАКИ vs ТАРГЕТ
premium              | mean_0=13224.25 | mean_1=16583.70 | p=0.0000 ***
premium_wo_term      | mean_0=10454.37 | mean_1=15575.35 | p=0.0000 ***
bonus_malus          | mean_0=   10.42 | mean_1=    9.43 | p=0.0000 ***
experience_year      | mean_0=    6.16 | mean_1=    5.21 | p=0.0000 ***
car_age              | ПРОПУЩЕНО (пустая группа)
car_year             | mean_0= 2005.73 | mean_1= 2006.22 | p=0.0000 ***
engine_volume        | mean_0= 2269.42 | mean_1= 2150.60 | p=0.0000 ***
engine_power         | mean_0=  101.55 | mean_1=   98.39 | p=0.1530 


In [12]:
print("Типы данных в NUM_COLS:")
for col in NUM_COLS:
    if col in df_pd.columns:
        dtype = df_pd[col].dtype
        n_non_numeric = pd.to_numeric(df_pd[col], errors='coerce').isna().sum() - df_pd[col].isna().sum()
        print(f"  {col:20s}: {dtype}  | нечисловых значений: {n_non_numeric}")

Типы данных в NUM_COLS:
  premium             : float64  | нечисловых значений: 0
  premium_wo_term     : float64  | нечисловых значений: 0
  bonus_malus         : object  | нечисловых значений: 40
  experience_year     : int64  | нечисловых значений: 0
  car_age             : object  | нечисловых значений: 569508
  car_year            : object  | нечисловых значений: 85
  engine_volume       : float64  | нечисловых значений: 0
  engine_power        : float64  | нечисловых значений: 0


In [13]:

# ============================================================
# 6. АНАЛИЗ КАТЕГОРИАЛЬНЫХ ПРИЗНАКОВ
# ============================================================
print("\n" + "=" * 50)
print("6. КАТЕГОРИАЛЬНЫЕ ПРИЗНАКИ")
print("=" * 50)

for col in CAT_COLS:
    if col not in df_pd.columns:
        continue
    n_unique = df_pd[col].nunique()
    claim_rate = df_pd.groupby(col)[TARGET].mean()
    print(f"\n{col} (уникальных: {n_unique}):")
    print(f"  TOP-5 по claim_rate:")
    print(claim_rate.sort_values(ascending=False).head(5).to_string())



6. КАТЕГОРИАЛЬНЫЕ ПРИЗНАКИ

region_name (уникальных: 22):
  TOP-5 по claim_rate:
region_name
04 Костанайская область              0.038614
16 Астана                            0.034186
05 Карагандинская область            0.025271
03 Восточно-Казахстанская область    0.024998
08 Павлодарская область              0.021091

vehicle_type_name (уникальных: 6):
  TOP-5 по claim_rate:
vehicle_type_name
Легковые                0.019938
Грузовые                0.012407
Автобусы до 16 п.м.     0.004450
Прицепы(полуприцепы)    0.003525
Автобусы > 16 п.м.      0.001963

mark (уникальных: 1524):
  TOP-5 по claim_rate:
mark
LADA 21902 111 41    1.0
ЛУИДОР               1.0
HYUNDAI   SOLARIS    1.0
SUBARU   FORESTER    1.0
TAYOTA CAMRY         1.0

model (уникальных: 11184):
  TOP-5 по claim_rate:
model
6460-001        1.0
212510 010      1.0
SPRINTER 410    1.0
VAZ210740       1.0
730 LI          1.0

age_experience_name (уникальных: 4):
  TOP-5 по claim_rate:
age_experience_name
менее 25 лет/стаж

In [14]:

# ============================================================
# 7. АНАЛИЗ BONUS-MALUS (важная переменная)
# ============================================================
print("\n" + "=" * 50)
print("7. BONUS-MALUS vs CLAIM RATE")
print("=" * 50)

bm_analysis = df_pd.groupby("bonus_malus").agg(
    count=(TARGET, "count"),
    claim_rate=(TARGET, "mean"),
    avg_premium=("premium", "mean")
).reset_index().sort_values("bonus_malus")
print(bm_analysis.to_string(index=False))



7. BONUS-MALUS vs CLAIM RATE
bonus_malus  count  claim_rate  avg_premium
          0     43    0.023256 29010.581395
          1    723    0.044260 27407.622407
         10  24605    0.020971 13440.707417
         11  27942    0.018753 12880.984718
         12  30842    0.017444 12484.261721
         13 301149    0.015368 12139.821155
          2   1252    0.035942 24366.496006
          3  29942    0.035502 16957.024681
          4  29828    0.026887 16101.248558
          5  23921    0.026504 15430.855984
          6  21236    0.025994 14900.031644
          7  26994    0.022857 14709.055568
          8  25655    0.021984 14225.816878
          9  25328    0.022781 13751.171628
          M     40    0.025000 34710.375000


In [15]:

# ============================================================
# 8. ВРЕМЕННОЙ АНАЛИЗ
# ============================================================
print("\n" + "=" * 50)
print("8. ВРЕМЕННОЙ АНАЛИЗ")
print("=" * 50)

df_pd["operation_date"] = pd.to_datetime(df_pd["operation_date"], errors="coerce")
df_pd["year_month"] = df_pd["operation_date"].dt.to_period("M")

monthly = df_pd.groupby("year_month").agg(
    policies=("contract_number", "nunique"),
    claim_rate=(TARGET, "mean")
).reset_index()
print(monthly.tail(12).to_string(index=False))



8. ВРЕМЕННОЙ АНАЛИЗ
year_month  policies  claim_rate
   2022-01     11866    0.017699
   2022-02     14523    0.020501
   2022-03     16283    0.019375
   2022-04     17115    0.018208
   2022-05     17898    0.019924
   2022-06     25728    0.017823
   2022-07     17524    0.020237
   2022-08     15729    0.020765
   2022-09     11139    0.019927
   2022-10     11191    0.020386
   2022-11     10690    0.021560
   2022-12     10949    0.018682


In [16]:

# ============================================================
# 9. IV / WoE ДЛЯ КЛЮЧЕВЫХ ПРИЗНАКОВ
# ============================================================
print("\n" + "=" * 50)
print("9. INFORMATION VALUE (IV)")
print("=" * 50)

def calc_iv(df, feature, target, bins=10):
    """Считает IV для числовой или категориальной переменной"""
    temp = df[[feature, target]].copy().dropna()
    
    if df[feature].dtype in ['object', 'category']:
        temp['bin'] = temp[feature]
    else:
        try:
            temp['bin'] = pd.qcut(temp[feature], q=bins, duplicates='drop')
        except Exception:
            return 0.0
    
    grouped = temp.groupby('bin')[target].agg(['sum', 'count'])
    grouped.columns = ['events', 'total']
    grouped['non_events'] = grouped['total'] - grouped['events']
    
    total_events = grouped['events'].sum()
    total_non_events = grouped['non_events'].sum()
    
    if total_events == 0 or total_non_events == 0:
        return 0.0
    
    grouped['dist_events'] = grouped['events'] / total_events
    grouped['dist_non_events'] = grouped['non_events'] / total_non_events
    
    grouped = grouped[(grouped['dist_events'] > 0) & (grouped['dist_non_events'] > 0)]
    grouped['woe'] = np.log(grouped['dist_events'] / grouped['dist_non_events'])
    grouped['iv'] = (grouped['dist_events'] - grouped['dist_non_events']) * grouped['woe']
    
    return grouped['iv'].sum()

iv_features = NUM_COLS + CAT_COLS + ["bonus_malus", "region_id"]
iv_results = {}
for col in iv_features:
    if col in df_pd.columns:
        iv = calc_iv(df_pd, col, TARGET)
        iv_results[col] = iv

iv_series = pd.Series(iv_results).sort_values(ascending=False)
print("\nIV по признакам (>0.02 = значимый):")
for feat, iv in iv_series.items():
    strength = "Strong" if iv > 0.5 else ("Medium" if iv > 0.1 else ("Weak" if iv > 0.02 else "Negligible"))
    print(f"  {feat:25s}: {iv:.4f}  [{strength}]")




9. INFORMATION VALUE (IV)

IV по признакам (>0.02 = значимый):
  premium_wo_term          : 0.5455  [Strong]
  model                    : 0.4295  [Medium]
  premium                  : 0.2190  [Medium]
  region_name              : 0.1096  [Medium]
  region_id                : 0.0983  [Weak]
  mark                     : 0.0979  [Weak]
  bonus_malus              : 0.0769  [Weak]
  vehicle_type_name        : 0.0333  [Weak]
  experience_year          : 0.0329  [Weak]
  car_year                 : 0.0293  [Weak]
  age_experience_name      : 0.0279  [Weak]
  engine_power             : 0.0271  [Weak]
  engine_volume            : 0.0184  [Negligible]
  car_age                  : 0.0008  [Negligible]
  is_residence_name        : 0.0000  [Negligible]
  is_individual_person_name: 0.0000  [Negligible]


In [17]:
# ============================================================
# 10. SCORE КОЛОНКИ — быстрый обзор
# ============================================================
print("\n" + "=" * 50)
print("10. SCORE КОЛОНКИ — ПРОПУСКИ И КОРРЕЛЯЦИЯ С ТАРГЕТОМ")
print("=" * 50)

score_analysis = pd.DataFrame({
    'missing_pct': df_pd[SCORE_COLS].isnull().mean() * 100,
    'corr_with_target': df_pd[SCORE_COLS].corrwith(df_pd[TARGET])
}).round(3)

score_analysis['abs_corr'] = score_analysis['corr_with_target'].abs()
print("\nТОП-10 SCORE колонок по корреляции с таргетом:")
print(score_analysis.sort_values('abs_corr', ascending=False).head(10).to_string())
print(f"\nSCORE колонок с пропусками > 50%: {(score_analysis['missing_pct'] > 50).sum()}")



10. SCORE КОЛОНКИ — ПРОПУСКИ И КОРРЕЛЯЦИЯ С ТАРГЕТОМ

ТОП-10 SCORE колонок по корреляции с таргетом:
            missing_pct  corr_with_target  abs_corr
SCORE_6_3         9.334            -0.016     0.016
SCORE_8_3         1.289             0.016     0.016
SCORE_6_2         9.334            -0.016     0.016
SCORE_7_1        64.949            -0.016     0.016
SCORE_11_1       85.438             0.016     0.016
SCORE_8_1         1.289             0.015     0.015
SCORE_11_2       85.438             0.014     0.014
SCORE_8_2         1.289             0.014     0.014
SCORE_2_1        15.503             0.014     0.014
SCORE_2_2        15.503            -0.013     0.013

SCORE колонок с пропусками > 50%: 22


In [18]:
# Сначала конвертируй bonus_malus в числа
df_pd['bonus_malus_num'] = pd.to_numeric(df_pd['bonus_malus'], errors='coerce')

score_aggs = {col: (col, 'mean') for col in SCORE_COLS if col in df_pd.columns}

policy_level = df_pd.groupby("contract_number").agg(
    is_claim=(TARGET, "max"),
    premium=("premium", "first"),
    premium_wo_term=("premium_wo_term", "first"),
    claim_amount=("claim_amount", "first"),
    
    driver_count=("driver_iin", "nunique"),
    min_experience=("experience_year", "min"),
    max_experience=("experience_year", "max"),
    
    # Используем числовую версию bonus_malus
    min_bonus_malus=("bonus_malus_num", "min"),
    max_bonus_malus=("bonus_malus_num", "max"),
    mean_bonus_malus=("bonus_malus_num", "mean"),
    # Оригинальный как категория (первый водитель)
    bonus_malus_raw=("bonus_malus", "first"),
    
    region_id=("region_id", "first"),
    region_name=("region_name", "first"),
    vehicle_type_id=("vehicle_type_id", "first"),
    car_age=("car_age", "first"),
    engine_power=("engine_power", "first"),
    engine_volume=("engine_volume", "first"),
    operation_date=("operation_date", "first"),
    mark=("mark", "first"),
    **score_aggs
).reset_index()

print(f"Shape: {policy_level.shape}")
print(f"Claim rate: {policy_level['is_claim'].mean():.2%}")

# Сколько bonus_malus стало NaN после конвертации (те самые "M" значения)
bm_nan_pct = policy_level['mean_bonus_malus'].isna().mean() * 100
print(f"bonus_malus NaN после конвертации: {bm_nan_pct:.1f}%")

policy_level.to_csv("policy_level_clean.csv", index=False)
print("Сохранено: policy_level_clean.csv")

Shape: (180635, 148)
Claim rate: 1.98%
bonus_malus NaN после конвертации: 0.0%
Сохранено: policy_level_clean.csv


In [21]:
# ============================================================
# 12. WoE АНАЛИЗ
# ============================================================
print("\n" + "=" * 50)
print("12. WoE АНАЛИЗ")
print("=" * 50)

def calc_woe_iv(df, feature, target, bins=10, cat=False):
    """
    Считает WoE и IV для признака.
    Возвращает таблицу по бинам + итоговый IV.
    """
    temp = df[[feature, target]].copy().dropna()
    
    if cat:
        temp['bin'] = temp[feature].astype(str)
    else:
        temp[feature] = pd.to_numeric(temp[feature], errors='coerce')
        temp = temp.dropna()
        try:
            temp['bin'] = pd.qcut(temp[feature], q=bins, duplicates='drop')
        except Exception:
            temp['bin'] = pd.cut(temp[feature], bins=bins, duplicates='drop')

    total_events     = (temp[target] == 1).sum()
    total_nonevents  = (temp[target] == 0).sum()

    grouped = temp.groupby('bin', observed=True)[target].agg(
        events=lambda x: (x == 1).sum(),
        non_events=lambda x: (x == 0).sum(),
        total='count'
    ).reset_index()

    grouped['event_rate']    = grouped['events'] / grouped['total']
    grouped['dist_events']   = grouped['events'] / total_events
    grouped['dist_nonevents']= grouped['non_events'] / total_nonevents

    # Защита от деления на ноль
    grouped['dist_events']    = grouped['dist_events'].replace(0, 0.0001)
    grouped['dist_nonevents'] = grouped['dist_nonevents'].replace(0, 0.0001)

    grouped['WoE'] = np.log(grouped['dist_events'] / grouped['dist_nonevents'])
    grouped['IV']  = (grouped['dist_events'] - grouped['dist_nonevents']) * grouped['WoE']

    iv_total = grouped['IV'].sum()
    return grouped, iv_total


# Признаки для WoE анализа
WOE_NUM_FEATURES = ['experience_year', 'engine_volume', 'engine_power', 'bonus_malus']
WOE_CAT_FEATURES = ['region_name', 'vehicle_type_name', 'age_experience_name', 'car_age', 'car_old']

all_iv = {}

# --- Числовые ---
print("\n[ ЧИСЛОВЫЕ ПРИЗНАКИ ]\n")
for col in WOE_NUM_FEATURES:
    if col not in df_pd.columns:
        continue
    woe_table, iv = calc_woe_iv(df_pd, col, TARGET, bins=10, cat=False)
    all_iv[col] = iv
    print(f"\n{'='*45}")
    print(f"Признак: {col}  |  IV = {iv:.4f}")
    print(f"{'='*45}")
    print(woe_table[['bin', 'total', 'events', 'event_rate', 'WoE', 'IV']].to_string(index=False))

# --- Категориальные ---
print("\n\n[ КАТЕГОРИАЛЬНЫЕ ПРИЗНАКИ ]\n")
for col in WOE_CAT_FEATURES:
    if col not in df_pd.columns:
        continue
    woe_table, iv = calc_woe_iv(df_pd, col, TARGET, cat=True)
    all_iv[col] = iv
    print(f"\n{'='*45}")
    print(f"Признак: {col}  |  IV = {iv:.4f}")
    print(f"{'='*45}")
    print(woe_table[['bin', 'total', 'events', 'event_rate', 'WoE', 'IV']]
          .sort_values('WoE', ascending=False).to_string(index=False))

# --- Итоговая таблица IV ---
print("\n\n" + "=" * 50)
print("ИТОГОВАЯ ТАБЛИЦА IV")
print("=" * 50)

iv_df = pd.DataFrame({
    'feature': list(all_iv.keys()),
    'IV': list(all_iv.values())
}).sort_values('IV', ascending=False)

def iv_strength(iv):
    if iv > 0.5:  return '🔴 Suspicious (leakage?)'
    if iv > 0.3:  return '🟢 Strong'
    if iv > 0.1:  return '🟡 Medium'
    if iv > 0.02: return '🟠 Weak'
    return '⚫ Negligible'

iv_df['strength'] = iv_df['IV'].apply(iv_strength)
print(iv_df.to_string(index=False))

# # Сохраняем для отчёта
# iv_df.to_csv("woe_iv_summary.csv", index=False)
# print("\nСохранено: woe_iv_summary.csv")


12. WoE АНАЛИЗ

[ ЧИСЛОВЫЕ ПРИЗНАКИ ]


Признак: experience_year  |  IV = 0.0329
           bin  total  events  event_rate       WoE       IV
 (-1.001, 0.0]  88935    2118    0.023815  0.205366 0.007279
    (0.0, 2.0]  52218    1294    0.024781  0.246100 0.006262
    (2.0, 3.0]  84726    1639    0.019345 -0.007106 0.000007
    (3.0, 4.0]  26506     645    0.024334  0.227455 0.002690
    (4.0, 6.0]  53273    1052    0.019747  0.013905 0.000018
    (6.0, 8.0]  49360     982    0.019895  0.021487 0.000040
   (8.0, 10.0] 190585    3007    0.015778 -0.214556 0.013917
(10.0, 2023.0]  23905     357    0.014934 -0.270364 0.002701

Признак: engine_volume  |  IV = 0.0184
              bin  total  events  event_rate       WoE       IV
 (-0.001, 1485.0]  58391    1189    0.020363  0.044163 0.000205
 (1485.0, 1591.0]  60055    1287    0.021430  0.096355 0.001028
 (1591.0, 1598.0]  59302    1300    0.021922  0.119526 0.001579
 (1598.0, 1797.0]  50081     927    0.018510 -0.053121 0.000242
 (1797.0,

In [20]:

# ============================================================
# ИТОГО
# ============================================================
print("\n" + "=" * 50)
print("EDA ЗАВЕРШЁН")
print("=" * 50)
print(f"  Текущий Loss Ratio:   {current_lr:.2%}")
print(f"  Дисбаланс таргета:   {target_pct[1]:.2f}% позитивных")
print(f"  Полисов для ML:       {len(policy_level):,}")
print(f"  Топ-признак по IV:    {iv_series.index[0]} ({iv_series.iloc[0]:.4f})")


EDA ЗАВЕРШЁН
  Текущий Loss Ratio:   123.34%
  Дисбаланс таргета:   1.95% позитивных
  Полисов для ML:       180,635
  Топ-признак по IV:    premium_wo_term (0.5455)
